# 01 — Exploratory Data Analysis Foundations

### Learn to understand data before asking a model to learn from it

**Level:** beginner → advanced  
**Based on:** the supplied EDA lecture transcript and dataset walkthroughs.

> **Simple picture:** data is a new toy box. EDA means opening the box slowly, checking what is inside, and making sure nothing is broken before we build something with it.

## Learning goals

By the end, you can explain what EDA is, profile a dataset, spot missing or duplicate records, choose useful plots, read correlations carefully, and separate EDA from feature engineering.


## 1. What is EDA?

**Exploratory Data Analysis (EDA)** is the first careful look at a dataset. We use summaries, tables, and plots to answer simple questions before building a machine-learning model.

- What does one row represent?
- Which columns are numbers, words, dates, or labels?
- Are values missing, duplicated, impossible, or strangely large?
- What patterns appear in one column, between two columns, or across many columns?

EDA does not mean “make lots of charts.” Every chart should help answer a question about the data.


## 2. The safe EDA recipe

1. **Read the problem.** Know what a row, feature, and target mean.
2. **Look at the shape.** Count rows and columns.
3. **Preview a few rows.** Check whether the values make sense.
4. **Check data types.** Numbers stored as text cause many hidden problems.
5. **Check missing values and duplicates.** Decide what they mean before removing or filling them.
6. **Summarise numbers.** Use count, mean, median, minimum, maximum, and spread.
7. **Plot distributions and relationships.** Ask focused questions.
8. **Write findings in words.** A model needs data, but people need an explanation.

### Little memory line

**Understand first, clean second, model later.**


In [ ]:
# Beginner-friendly guide:
# These libraries help us inspect tables, calculate summaries, and draw clear pictures.
# Importing them once at the top keeps later cells short and easy to read.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# This makes plots easier to read in a notebook.
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)


## 3. A tiny practice dataset

The small table below lets us practise the EDA recipe. In a real project, replace it with your own CSV or Excel file.

Notice that `NaN` means a value is missing. It is not the same as zero.


In [ ]:
# Beginner-friendly guide:
# We create a tiny pretend dataset with numbers, categories, a missing value, and a duplicate row.
# A small example makes it safe to learn each EDA check before using a large real dataset.
toy = pd.DataFrame({
    "age": [22, 35, np.nan, 35, 45, 35],
    "city": ["Delhi", "Mumbai", "Delhi", "Mumbai", "Chennai", "Mumbai"],
    "spend": [120, 340, 210, 340, 5000, 340],
    "bought": [0, 1, 0, 1, 1, 1],
})

# head shows the first few rows, while shape tells us how many rows and columns exist.
print("Shape (rows, columns):", toy.shape)
display(toy.head())


In [ ]:
# Beginner-friendly guide:
# This helper collects the first health checks we should run on almost every table.
# It reports column types, missing values, duplicate rows, and a number summary without changing the data.
def quick_profile(df):
    print("Shape:", df.shape)
    print("\nColumn types:")
    display(df.dtypes.rename("data_type"))
    print("\nMissing values:")
    display(df.isna().sum().rename("missing_count"))
    print("\nDuplicate rows:", df.duplicated().sum())
    print("\nNumeric summary:")
    display(df.describe(include="number").T)

# Run the same checklist on the practice table.
quick_profile(toy)


## 4. Missing values, duplicates, and outliers

### Missing values

A blank can mean a person skipped a question, a sensor failed, or a value does not apply. First learn **why** it is missing. Then choose a response: keep it, fill it, add a missing-value flag, or remove a row/column when that is safe.

### Duplicates

Two identical rows may be an accidental copy, or they may be two real repeated events. Check the meaning of a row before deleting duplicates.

### Outliers

An outlier is a value far from most others. It might be a typing error, or it might be an important real case. Investigate before clipping or removing it.


In [ ]:
# Beginner-friendly guide:
# We count blanks and duplicate rows, then use the IQR rule to flag unusually far-away numeric values.
# A flag is only a question mark; it is not permission to delete a real observation automatically.
print("Missing values by column:\n", toy.isna().sum())
print("\nDuplicate rows:", toy.duplicated().sum())

q1, q3 = toy["spend"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
flagged = toy.loc[(toy["spend"] < lower) | (toy["spend"] > upper)]

print(f"\nIQR limits: {lower:.1f} to {upper:.1f}")
display(flagged)


## 5. Univariate, bivariate, and multivariate analysis

| Name | Easy question | Useful tools |
| --- | --- | --- |
| **Univariate** | “What does one column look like?” | Histogram, box plot, bar chart. |
| **Bivariate** | “How do two columns move together?” | Scatter plot, grouped box plot, cross-tabulation. |
| **Multivariate** | “What happens when several columns join the story?” | Heatmap, pair plot, grouped chart. |

### Correlation is not a magic answer

Correlation tells whether two numeric columns move together in a roughly straight-line way. It does **not** prove that one column causes the other. A hidden third factor can explain both.


In [ ]:
# Beginner-friendly guide:
# The left plot looks at one number column. The right plot compares spending with the yes-or-no target.
# The heatmap then gives a quick view of how numeric columns move together.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(toy["spend"], bins=5, kde=True, ax=axes[0])
axes[0].set_title("One column: spend")

sns.boxplot(data=toy, x="bought", y="spend", ax=axes[1])
axes[1].set_title("Two columns: spend by outcome")

sns.heatmap(toy.select_dtypes("number").corr(), annot=True, cmap="coolwarm", center=0, ax=axes[2])
axes[2].set_title("Numeric correlations")

plt.tight_layout()


## 6. EDA versus feature engineering

EDA is the **detective work**: understand what is already in the dataset. Feature engineering is the **preparation work**: create or transform useful inputs for a model.

Examples of feature engineering:

- Split a date into day, month, and weekday.
- Turn `"2 stops"` into the number `2`.
- Convert a duration like `"2h 30m"` into 150 minutes.
- Convert categories into model-friendly columns.
- Apply a log transform to a very long right tail.

Do EDA before and after feature engineering. That helps you confirm that a transformation did what you expected.


## End-of-topic checklist

Before modelling, you should be able to say:

- What one row represents.
- Which column is the target, if there is one.
- Which columns need cleaning or type conversion.
- Where values are missing or duplicated.
- Which patterns are interesting and which are only weak clues.
- Which transformations are safe to learn from training data only.

**Best habit:** write down your findings. A clear sentence beside a chart is more useful than a chart with no question behind it.
